# The Machine-Translation Dataset

In [1]:
import os
import sys
import pickle
import torch
import torch.optim as optim
import torch.nn as nn
from torchtext.vocab import GloVe

PROJECT_DIR = os.path.expanduser("~/Vietnamese-Poem-Generation/")
TRANSLATION_UTILS_DIR = os.path.join(PROJECT_DIR, "translation/utils")
TRANSLATION_MODELS_DIR = os.path.join(PROJECT_DIR, "translation/models")
sys.path.append(PROJECT_DIR)
sys.path.append(TRANSLATION_UTILS_DIR)
sys.path.append(TRANSLATION_MODELS_DIR)
from constants import TRANSLATION_DATA_DIR, TRANSLATION_TRAIN_DIR, \
                      TRANSLATION_VAL_DIR, TRANSLATION_TEST_DIR, \
                      STORAGE_DIR, SRC_LANGUAGE, TGT_LANGUAGE, \
                      PAD_IDX

from data_splitting import split_and_save_data
from tokenization import build_vocabulary
from data_loader import get_dataloader
from network import *
from train import train_model

2025-02-11 17:08:12.495988: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1739268492.575763    5571 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1739268492.599601    5571 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2025-02-11 17:08:12.952730: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


## Overview

In [2]:
en_data_dir = os.path.join(TRANSLATION_DATA_DIR, "TED2020.en-vi.en")
vi_data_dir = os.path.join(TRANSLATION_DATA_DIR, "TED2020.en-vi.vi")

In [3]:
en_data = open(en_data_dir, "r").readlines()
vi_data = open(vi_data_dir, "r").readlines()

In [4]:
en_data = [line.strip() for line in en_data]
vi_data = [line.strip() for line in vi_data]

In [5]:
(len(en_data), len(vi_data))

(326417, 326417)

In [6]:
en_data[100], vi_data[100]

("And if they judge how much they're going to get paid on your capital that they've invested, based on the short-term returns, you're going to get short-term decisions.",
 'Và nếu họ xem xét việc họ sẽ được trả bao nhiêu tiền dựa trên số vốn của bạn mà họ đã đầu tư, dựa trên những món hoàn trả ngắn hạn, bạn sẽ nhận được những quyết định ngắn hạn.')

In [7]:
max_en_len = max(len(line.split()) for line in en_data)
max_vi_len = max(len(line.split()) for line in vi_data)
print("Max English sentence length:", max_en_len, "\nMax Vietnamese sentence length:", max_vi_len)

Max English sentence length: 625 
Max Vietnamese sentence length: 838


## Data Splitting

In [ ]:
split_and_save_data(
    source_file=os.path.join(TRANSLATION_DATA_DIR, "TED2020.en-vi.vi"),
    target_file=os.path.join(TRANSLATION_DATA_DIR, "TED2020.en-vi.en")
)

## Tokenization

In [8]:
vocab_transform = build_vocabulary(
    source_file=os.path.join(TRANSLATION_DATA_DIR, "TED2020.en-vi.vi"),
    target_file=os.path.join(TRANSLATION_DATA_DIR, "TED2020.en-vi.en")
)

In [ ]:
len(vocab_transform[SRC_LANGUAGE].get_itos()), len(vocab_transform[TGT_LANGUAGE].get_itos())

(38626, 221188)

In [ ]:
vocab_transform[SRC_LANGUAGE].get_itos()[:10], vocab_transform[TGT_LANGUAGE].get_itos()[:10]

(['<unk>', '<pad>', '<bos>', '<eos>', ',', '.', 'và', 'tôi', 'là', 'có'],
 ['<unk>', '<pad>', '<bos>', '<eos>', ',', '.', 'the', 'to', 'that', 'in'])

Save the vocabulary:

In [24]:
with open(os.path.join(STORAGE_DIR, "translation_vocab_transform.pkl"), "wb") as f:
    pickle.dump(vocab_transform, f)

Load the vocabulary:

In [2]:
with open(os.path.join(STORAGE_DIR, "translation_vocab_transform.pkl"), "rb") as f:
    vocab_transform = pickle.load(f)

## Data Loader

In [3]:
train_loader = get_dataloader(
    source_file=os.path.join(TRANSLATION_TRAIN_DIR, "train.vi"),
    target_file=os.path.join(TRANSLATION_TRAIN_DIR, "train.en"),
    vocab_transform=vocab_transform,
    batch_size=32,
    mode="train"
)

val_loader = get_dataloader(
    source_file=os.path.join(TRANSLATION_VAL_DIR, "val.vi"),
    target_file=os.path.join(TRANSLATION_VAL_DIR, "val.en"),
    vocab_transform=vocab_transform,
    batch_size=16,
    mode="val"
)

test_loader = get_dataloader(
    source_file=os.path.join(TRANSLATION_TEST_DIR, "test.vi"),
    target_file=os.path.join(TRANSLATION_TEST_DIR, "test.en"),
    vocab_transform=vocab_transform,
    batch_size=16,
    mode="test"
)

In [11]:
src_ids, tgt_ids = next(iter(train_loader))
src_ids.shape, tgt_ids.shape

(torch.Size([32, 41]), torch.Size([32, 45]))

In [12]:
src_ids, tgt_ids

(tensor([[    2, 10310, 19892,  ...,     1,     1,     1],
         [    2,  1934,  3006,  ...,     1,     1,     1],
         [    2,   515,  1227,  ...,     1,     1,     1],
         ...,
         [    2,  6179,   482,  ...,     1,     1,     1],
         [    2,    70,   806,  ...,     1,     1,     1],
         [    2,  7381,   187,  ...,     1,     1,     1]]),
 tensor([[    2,     0, 24915,  ...,   211,   211,     3],
         [    2,  8618,     4,  ...,     1,     1,     1],
         [    2,     0,     4,  ...,     1,     1,     1],
         ...,
         [    2,     0,     0,  ...,     1,     1,     1],
         [    2,    39,  1121,  ...,     1,     1,     1],
         [    2,     0,    17,  ...,     1,     1,     1]]))

## Training

In [4]:
input_size = len(vocab_transform[SRC_LANGUAGE])
output_size = len(vocab_transform[TGT_LANGUAGE])
hidden_size = 300

In [15]:
print(f"Input size: {input_size}, Output size: {output_size}, Hidden size: {hidden_size}")

Input size: 38626, Output size: 221188, Hidden size: 300


In [5]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
criterion = nn.CrossEntropyLoss(ignore_index=PAD_IDX)

In [6]:
glove = GloVe(name="6B", dim=300)  # Use 300-dimensional GloVe embedding
pretrained_embedding = glove.vectors

model = Seq2Seq_GRU(
    encoder=EncoderGRU(input_size, hidden_size, pretrained_embedding=pretrained_embedding, freeze_embedding=True),
    decoder=DecoderGRU(hidden_size, output_size, pretrained_embedding=pretrained_embedding, freeze_embedding=True),
    device=device
).to(device)

In [7]:
optimizer = optim.Adam(model.parameters())
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode="min", factor=0.1, patience=10)

In [13]:
train_model(
    model=model,
    train_loader=train_loader,
    val_loader=val_loader,
    criterion=criterion,
    optimizer=optimizer,
    scheduler=scheduler,
    num_epochs=1,
    device=device,
    model_name="seq2seq_gru",
)

Epoch 1/1:   0%|          | 0/8161 [00:00<?, ?it/s]


OutOfMemoryError: CUDA out of memory. Tried to allocate 28.00 MiB (GPU 0; 5.70 GiB total capacity; 4.93 GiB already allocated; 44.38 MiB free; 5.08 GiB reserved in total by PyTorch) If reserved memory is >> allocated memory try setting max_split_size_mb to avoid fragmentation.  See documentation for Memory Management and PYTORCH_CUDA_ALLOC_CONF

# The Poem Dataset